# ScholarLens — Host LLM (vLLM) trên Kaggle + public qua trycloudflare

App chạy trên máy bạn (Docker); notebook này biến Kaggle thành **máy GPU phục vụ LLM** (API tương thích OpenAI),
và tuỳ chọn chạy luôn **embedding + đồ thị** cho kho dữ liệu.

**Chuẩn bị (1 lần):**
1. Trên máy bạn: `python scripts/make_kaggle_bundle.py --with-data` → `dist/scholarlens-code-data.zip`.
2. Kaggle → **Datasets → New Dataset** → kéo file zip vào → tên `scholarlens-code` → **Create**.
3. Mở notebook này trên Kaggle → **Add Input** → chọn dataset `scholarlens-code`.
4. **Settings**: Accelerator = **GPU T4 x2**, Internet = **On**. (Tuỳ chọn) **Add-ons → Secrets**: `VLLM_API_KEY` (khoá cố định), `HF_TOKEN` (model gated như Llama).
5. **Run All**. Ô cuối in **URL trycloudflare + API key** → dán vào nút LLM trên app.

> Phiên Kaggle tương tác tối đa ~12 giờ và tính vào hạn mức GPU theo tuần; tắt phiên (Stop session) khi demo xong.

In [ ]:
# ===== Cấu hình =====
PRESET        = "qwen2.5-7b-awq"   # qwen2.5-7b-awq (2xT4) | qwen2.5-3b | qwen2.5-1.5b | llama3.1-8b-awq | qwen2.5-14b-awq
MODEL         = ""                 # hoặc repo HF bất kỳ vLLM hỗ trợ, ví dụ "Qwen/Qwen2.5-7B-Instruct-AWQ" (ghi đè PRESET)
MAX_MODEL_LEN = 16384
RUN_EMBEDDING = True               # embedding + graph cho data/ trong dataset (chạy trước vLLM để không tranh VRAM)
VLLM_VERSION  = ""                 # "" = bản mới nhất; nếu lỗi trên T4 thử ghim, ví dụ "0.29.0"

In [ ]:
import os, glob, shutil, subprocess, sys
# Tìm thư mục chứa mã nguồn trong /kaggle/input (dataset đã Add Input)
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/scholarlens/__init__.py", recursive=True)]
assert cands, "Không thấy mã nguồn: hãy Add Input dataset scholarlens-code (xem hướng dẫn ở trên)"
SRC, WORK = cands[0], "/kaggle/working/ScholarLens"
shutil.copytree(SRC, WORK, dirs_exist_ok=True)          # /kaggle/input là read-only
os.chdir(WORK); sys.path.insert(0, WORK)
os.environ["PYTHONIOENCODING"] = "utf-8"
print("Mã nguồn:", SRC, "->", WORK)

# Secrets (tuỳ chọn)
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    for name in ("VLLM_API_KEY", "HF_TOKEN"):
        try:
            os.environ[name] = sec.get_secret(name); print("Đã nạp secret", name)
        except Exception:
            pass
except ImportError:
    pass
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv

## 1. (Tuỳ chọn) Embedding + đồ thị trên GPU
Kết quả: `scholarlens_data.zip` ở tab **Output** → tải về, giải nén thành `data/` cạnh `docker-compose.yml`.

In [ ]:
if RUN_EMBEDDING and os.path.exists("data/scholarlens.sqlite"):
    !pip install -q -r requirements-app.txt
    os.environ["SL_EMBED_DEVICE"] = "cuda"
    !python -m scholarlens.embedding --device cuda
    !python -m scholarlens.graph build
    !python scripts/evaluate.py --n 500 --models intfloat/multilingual-e5-base BAAI/bge-base-en-v1.5
    shutil.copy("reports/eval_results.json", "/kaggle/working/eval_results.json") if os.path.exists("reports/eval_results.json") else None
    shutil.make_archive("/kaggle/working/scholarlens_data", "zip", "data")
    !ls -lh /kaggle/working/scholarlens_data.zip /kaggle/working/eval_results.json
else:
    print("Bỏ qua embedding (RUN_EMBEDDING=False hoặc dataset không kèm data/)")

## 2. Cài vLLM

In [ ]:
spec = f"vllm=={VLLM_VERSION}" if VLLM_VERSION else "vllm"
!pip install -q {spec} openai python-dotenv
# Kaggle cài sẵn torchaudio build cho CUDA cũ; vllm nâng torch lên CUDA mới -> transformers import torchaudio
# và dừng với lỗi "PyTorch and TorchAudio were compiled with different CUDA versions".
# Model văn bản không cần torchaudio nên gỡ đi.
!pip uninstall -y -q torchaudio
# Kiểm tra trước đúng module server sẽ chạy (lỗi môi trường hiện ra ở đây thay vì sau vài phút chờ)
!python -c "import torch, vllm; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| vLLM', vllm.__version__); import vllm.entrypoints.openai.api_server; print('vLLM import OK')"


## 3. Khởi động vLLM + tunnel trycloudflare
Lần đầu tải trọng số model từ Hugging Face mất vài phút. Log chi tiết: `data/llm_server/vllm.log`.

In [ ]:
import importlib, scholarlens.llm_server.server as srv
importlib.reload(srv)
p = srv.PRESETS.get(PRESET, {})
cfg = srv.ServerConfig(model=MODEL or p["model"], max_model_len=MAX_MODEL_LEN,
                       quantization="" if MODEL else p.get("quantization", ""),
                       gpu_memory_utilization=0.90)
info = srv.serve(cfg, tunnel="cloudflared", test=True, detach=True)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(f"""
### Dán vào nút **LLM** trên app (góc phải ô chat)
| | |
|---|---|
| Provider | OpenAI-compatible |
| Base URL | `{info['public_url']}` |
| Model | `{info['model']}` |
| API key | `{info['api_key']}` |
"""))

## 4. Thử gọi qua URL công khai (giống cách app sẽ gọi)

In [ ]:
from openai import OpenAI
c = OpenAI(base_url=info["public_url"], api_key=info["api_key"])
r = c.chat.completions.create(model=info["model"], max_tokens=120, stream=True,
        messages=[{"role": "user", "content": "Tóm tắt Graph RAG trong 2 câu."}])
for ch in r:
    print(ch.choices[0].delta.content or "", end="")

## 5. Giữ phiên sống
Để ô này chạy suốt buổi demo. Mỗi 2 phút kiểm tra vLLM và tunnel; nếu tunnel chết sẽ tự mở tunnel mới và in **URL MỚI** (dán lại vào app). Dừng: nút ■ Interrupt.

In [ ]:
srv.keep_alive(info, every=120, max_hours=11.5)